# Ligamento · E1 — cabezas mixtas softmax+delta (Colab Pro)

Campaña principal de E1 (§6 del protocolo v1.0 congelado). Condiciones **C1** (softmax), **C2** (delta),
**C3** (mix22 = 2 softmax + 2 delta) y **C4** (mix31/mix13, exploratorias), **8 semillas**, tareas **T1**
(capacidad, 6 cargas) + **T2** (correctabilidad).

**Convergencia (decisión de Maxi 2026-07-22):** acc@1 en {L96, L128} cada 500 pasos; converge cuando la
mejora en la ventana de 500 es < 0.5 pts; bloques uniformes de +2500 con **tope duro 10 000**. Las 8 semillas
de cada condición bajo el **mismo régimen**.

**Uso:** `Entorno de ejecución → Cambiar tipo de entorno → GPU`, luego `Ejecutar todo`.
Si la sesión se corta, **volvé a ejecutar todo**: reanuda desde los checkpoints de Drive.

In [ ]:
# 1) Hardware. En Colab Pro esperamos L4 / A100 (T4 tambien sirve, mas lento).
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'SIN GPU: Entorno de ejecucion -> Cambiar tipo -> GPU'

In [ ]:
# 2) Dependencias
!pip -q install optax >/dev/null
import jax; print('jax', jax.__version__, '| devices:', jax.devices())
assert jax.devices()[0].platform == 'gpu', 'No hay GPU visible para JAX. Activa GPU en el entorno.'

In [ ]:
# 3) Codigo pre-registrado (repo publico)
![ -d telar-ligamento ] && (cd telar-ligamento && git pull -q) || git clone -q https://github.com/SperanzaMax/telar-ligamento
!ls telar-ligamento/experimentos/E1

In [ ]:
# 4) Drive: checkpoints + resultados sobreviven al corte de sesion
from google.colab import drive
drive.mount('/content/drive')
import os; RESULTS='/content/drive/MyDrive/ligamento_e1'; os.makedirs(RESULTS, exist_ok=True)
print('resultados ->', RESULTS)

## Gate de pre-registro

E1 **no se lanza** si el protocolo madre o el prereg de seguimiento congelado no coinciden con su hash.
El `--requiere PREREG_SEGUIMIENTO_v1.1` implementa la **condición 3** del plan de arranque: el prereg de
seguimiento actualizado tiene que estar congelado y pusheado antes de correr.

In [ ]:
# 5) Verificar anclas (bloquea si algo cambio o si falta el prereg v1.1 congelado)
!cd telar-ligamento && python experimentos/verificar_anclas.py --requiere PREREG_SEGUIMIENTO_v1.1

In [ ]:
# 6) (opcional) Costo real en ESTA GPU antes de comprometer horas de Pro.
#    No toca ninguna metrica de tarea: solo cronometra el paso y proyecta la campania.
!cd telar-ligamento && python experimentos/E1/bench_costo.py 30

In [ ]:
# 7) (opcional) Aviso por Telegram al terminar cada bloque. El token va SOLO por env, nunca al repo.
import os
os.environ['TG_TOKEN'] = ''        # <- pegar token aqui si se quiere aviso; vacio = sin avisos
os.environ['TG_CHAT']  = '7985522502'
print('avisos Telegram:', 'ON' if os.environ['TG_TOKEN'] else 'OFF')

## Celda maestra (reanudable)

Corre las 4 condiciones en orden. **C2 (delta) primero**: es la que instancia los márgenes R11 y la carga de
evaluación del prereg, y la que más pasos necesita. Si la sesión se corta, re-ejecutar esta celda retoma
desde el último checkpoint de cada semilla (determinista: reanudar == correr de corrido).

In [ ]:
# 8) E1 completo (C2 -> C1 -> C3 -> C4). Reanudable: re-ejecutar saltea lo ya hecho.
%env RESULTS_DIR=/content/drive/MyDrive/ligamento_e1
%env N_SEEDS=8
%env CONDS=delta,softmax,mix22,mix31,mix13
!cd telar-ligamento/experimentos/E1 && python e1_runner.py

In [ ]:
# 9) Informe agregado (veredictos preliminares P1.1 / P1.2 / P1.3 + PS-1)
import os
p = '/content/drive/MyDrive/ligamento_e1/E1_informe.md'
print(open(p).read() if os.path.exists(p) else 'aun no generado (la campania no termino)')

In [ ]:
# 10) Estado de avance por semilla (util si la sesion se corto a mitad)
import glob, json, os
for f in sorted(glob.glob('/content/drive/MyDrive/ligamento_e1/e1_*.json')):
    d = json.load(open(f))
    print(f"{d['cond']:<8} s{d['seed']} @{d['steps']:>5} conv={str(d['converged']):<5} "
          f"L96={d['capacity']['96']['1']:.3f} L128={d['capacity']['128']['1']:.3f} T2={d['T2_L32']:.3f}")
ck = glob.glob('/content/drive/MyDrive/ligamento_e1/*.ckpt')
print(f"\ncheckpoints presentes: {len(ck)}")